# 01 — ML Pipeline Walkthrough

End-to-end mortality prediction on WHAS500 and UCI Heart Disease.

**Covers**: data loading, preprocessing, model training, evaluation, comparison.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from src.data.extract import load_both_datasets
from src.data.transform import prepare_whas500_for_ml, prepare_uci_for_ml
from src.models.train import train_all_models, build_models
from src.models.evaluate import compare_models
from src.utils.config import Paths, WHAS500Config

Paths.ensure_all()
print('Setup OK')

## 2. Load and Explore Data

In [ ]:
whas_raw, uci_raw = load_both_datasets()

print('WHAS500:')
print(f'  {len(whas_raw)} patients, {whas_raw["fstat"].mean():.1%} mortality')
print(f'  Median follow-up: {whas_raw["lenfol"].median():.0f} days')
print()
print('UCI Heart Disease:')
print(f'  {len(uci_raw)} patients, {uci_raw["target"].mean():.1%} with disease')
whas_raw.describe().round(2)

## 3. Preprocess and Split

In [ ]:
X_train_w, X_test_w, y_train_w, y_test_w, scaler_w = prepare_whas500_for_ml(whas_raw)
X_train_u, X_test_u, y_train_u, y_test_u, scaler_u = prepare_uci_for_ml(uci_raw)

print(f'WHAS500: {len(X_train_w)} train, {len(X_test_w)} test')
print(f'UCI:     {len(X_train_u)} train, {len(X_test_u)} test')
print(f'\nClass balance (WHAS500 train): {y_train_w.mean():.1%} positive')

## 4. Train All Models

In [ ]:
print('Training on WHAS500...')
results_w = train_all_models(X_train_w, X_test_w, y_train_w, y_test_w, dataset='whas500')

print('\nTraining on UCI Heart Disease...')
results_u = train_all_models(X_train_u, X_test_u, y_train_u, y_test_u, dataset='uci')

## 5. Compare Models

In [ ]:
from src.models.train import load_model

# Load fitted models
models_w = {name: load_model(name, 'whas500') for name in results_w}

comparison = compare_models(models_w, X_test_w, y_test_w)
print('WHAS500 — Model Comparison:')
print(comparison[['model', 'roc_auc', 'average_precision', 'f1', 'recall']].to_string(index=False))

## 6. ROC Curves

In [ ]:
from src.visualisation.plots import plot_roc_curves

fig = plot_roc_curves(models_w, X_test_w, y_test_w,
                       title='ROC Curves — WHAS500 Mortality Prediction')
plt.show()